<a href="https://colab.research.google.com/github/Serx17/compliance-checker-230fz/blob/main/notebooks/01_mvp_compliance_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Импорты и подготовка окружения
import json
import os
from datetime import datetime
from typing import List, Dict

# 2. Структурируем ключевые ограничения 230-ФЗ в JSON-совместимый формат
RULES_230FZ = {
    "law_title": "Федеральный закон № 230-ФЗ от 03.07.2016",
    "restrictions": [
        {
            "article": "ст. 7, ч. 5",
            "rule": "Запрет на взаимодействие в ночное время (22:00–08:00 в будни, 20:00–09:00 в выходные)",
            "keywords": ["ночью", "в 23", "в 3 ночи", "утром до 8", "поздно вечером"]
        },
        {
            "article": "ст. 7, ч. 4",
            "rule": "Ограничение частоты: не более 2 раз в неделю, не более 8 раз в месяц",
            "keywords": ["опять звоните", "каждый день", "третий раз за неделю", "постоянно"]
        },
        {
            "article": "ст. 6, ч. 1",
            "rule": "Запрет на угрозы, оскорбления, раскрытие долга третьим лицам",
            "keywords": ["коллекторы придут", "испортим кредитную историю", "расскажем работодателю", "суд", "арест имущества"]
        }
    ]
}

# Сохраняем локально в Colab (имитация базы знаний)
with open("230fz_rules.json", "w", encoding="utf-8") as f:
    json.dump(RULES_230FZ, f, ensure_ascii=False, indent=2)

print("✅ База правил сохранена в 230fz_rules.json")

# 3. Функция-формирователь промпта (пока без API-вызова)
def build_compliance_prompt(text: str, rules: Dict) -> str:
    rules_text = "\n".join([f"- {r['article']}: {r['rule']}" for r in rules["restrictions"]])

    prompt = f"""Ты — юридический ассистент по взысканию задолженности в РФ.
Проверь следующий текст коммуникации с должником на соответствие 230-ФЗ.
{rules["law_title"]}. Основные ограничения:
{rules_text}

Текст для проверки: "{text}"

Верни ответ ТОЛЬКО в формате JSON:
{{
  "compliant": true/false,
  "violations": [
    {{
      "article": "статья",
      "reason": "почему нарушено",
      "severity": "low/medium/high"
    }}
  ],
  "recommendation": "как исправить текст"
}}
"""
    return prompt

# Тестовый вызов
sample_sms = "Здравствуйте! Напоминаем о задолженности. Если не оплатите до 15 числа, передадим дело коллекторам и испортим КИ. Звонок в 23:30."
print(build_compliance_prompt(sample_sms, RULES_230FZ))

✅ База правил сохранена в 230fz_rules.json
Ты — юридический ассистент по взысканию задолженности в РФ.
Проверь следующий текст коммуникации с должником на соответствие 230-ФЗ.
Федеральный закон № 230-ФЗ от 03.07.2016. Основные ограничения:
- ст. 7, ч. 5: Запрет на взаимодействие в ночное время (22:00–08:00 в будни, 20:00–09:00 в выходные)
- ст. 7, ч. 4: Ограничение частоты: не более 2 раз в неделю, не более 8 раз в месяц
- ст. 6, ч. 1: Запрет на угрозы, оскорбления, раскрытие долга третьим лицам

Текст для проверки: "Здравствуйте! Напоминаем о задолженности. Если не оплатите до 15 числа, передадим дело коллекторам и испортим КИ. Звонок в 23:30."

Верни ответ ТОЛЬКО в формате JSON:
{
  "compliant": true/false,
  "violations": [
    {
      "article": "статья",
      "reason": "почему нарушено",
      "severity": "low/medium/high"
    }
  ],
  "recommendation": "как исправить текст"
}



In [ ]:
# 0. Установка зависимостей (выполни один раз в сессии)
!pip install pydantic requests -q

# 1. Импорт и конфигурация
import json
import requests
from typing import List
from pydantic import BaseModel, ValidationError

# 2. База знаний (перенесена из Шага 1 для целостности)
RULES_230FZ = {
    "law_title": "Федеральный закон № 230-ФЗ от 03.07.2016",
    "restrictions": [
        {"article": "ст. 7, ч. 5", "rule": "Запрет на взаимодействие в ночное время (22:00–08:00 будни, 20:00–09:00 выходные)", "keywords": ["ночью", "в 23", "в 3 ночи", "утром до 8"]},
        {"article": "ст. 7, ч. 4", "rule": "Ограничение частоты: не более 2 раз в неделю, не более 8 раз в месяц", "keywords": ["опять звоните", "каждый день", "третий раз за неделю"]},
        {"article": "ст. 6, ч. 1", "rule": "Запрет на угрозы, оскорбления, раскрытие долга третьим лицам", "keywords": ["коллекторы придут", "испортим кредитную историю", "расскажем работодателю"]}
    ]
}

# 3. Промпт-билдер (Contract-driven)
def build_compliance_prompt(text: str, rules: dict) -> str:
    rules_text = "\n".join([f"- {r['article']}: {r['rule']}" for r in rules["restrictions"]])
    return f"""Ты — юридический ассистент по взысканию задолженности в РФ.
Проверь текст коммуникации на соответствие {rules["law_title"]}.
Ограничения:
{rules_text}

Текст для проверки: "{text}"

Верни ответ СТРОГО в формате JSON (без markdown-обёрток и лишних символов):
{{
  "compliant": true/false,
  "violations": [
    {{
      "article": "статья закона",
      "reason": "конкретная фраза из текста, которая нарушает закон",
      "severity": "low/medium/high"
    }}
  ],
  "recommendation": "как переписать текст, чтобы он стал compliant"
}}
"""

# 4. API Caller (OpenRouter)
def call_llm(prompt: str, api_key: str) -> dict:
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
        "HTTP-Referer": "https://github.com/your-username/compliance-checker-230fz",
        "X-Title": "Compliance-Checker-MVP"
    }
    payload = {
        # Меняем модель на стабильную Llama 3 (бесплатный тир)
        "model": "meta-llama/llama-3-8b-instruct:free",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "response_format": {"type": "json_object"}
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()  # Автоматически выбросит ошибку при 4xx/5xx
    return response.json()

# 5. Валидация схемы (Pydantic)
class Violation(BaseModel):
    article: str
    reason: str
    severity: str

class ComplianceResult(BaseModel):
    compliant: bool
    violations: List[Violation]
    recommendation: str

def parse_and_validate_response(api_response: dict) -> ComplianceResult:
    raw_content = api_response["choices"][0]["message"]["content"]
    clean_content = raw_content.strip().removeprefix("```json").removesuffix("```").strip()

    try:
        return ComplianceResult.model_validate_json(clean_content)
    except ValidationError as e:
        print(f"⚠️ Ошибка валидации JSON: {e}")
        # Fallback для продакшена: не роняем пайплайн, логируем ошибку
        return ComplianceResult(compliant=False, violations=[], recommendation="Ошибка парсинга ответа модели. Требуется ручной аудит.")

# 6. Тестовый запуск (раскомментируй для проверки)
"""
API_KEY = input("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")
test_sms = "Здравствуйте! Напоминаем о долге. Если не заплатите до пятницы, передадим коллекторам и испортим КИ. Ждите звонка в 23:00."

prompt = build_compliance_prompt(test_sms, RULES_230FZ)
api_resp = call_llm(prompt, API_KEY)
result = parse_and_validate_response(api_resp)

print("\n✅ Результат проверки:")
print(f"Compliant: {result.compliant}")
for v in result.violations:
    print(f"🚨 {v.article} | {v.severity.upper()} | {v.reason}")
print(f"💡 Рекомендация: {result.recommendation}")
"""

'\nAPI_KEY = input("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")\ntest_sms = "Здравствуйте! Напоминаем о долге. Если не заплатите до пятницы, передадим коллекторам и испортим КИ. Ждите звонка в 23:00."\n\nprompt = build_compliance_prompt(test_sms, RULES_230FZ)\napi_resp = call_llm(prompt, API_KEY)\nresult = parse_and_validate_response(api_resp)\n\nprint("\n✅ Результат проверки:")\nprint(f"Compliant: {result.compliant}")\nfor v in result.violations:\n    print(f"🚨 {v.article} | {v.severity.upper()} | {v.reason}")\nprint(f"💡 Рекомендация: {result.recommendation}")\n'

In [ ]:
import requests
import time
from typing import List
from pydantic import BaseModel, ValidationError

# ==========================================
# 1. КОНФИГУРАЦИЯ
# ==========================================
USE_MOCK = True
API_KEY = "sk-or-v1-..."

# Списки слов вынесены в переменные (избегаем SyntaxError при переносе строк)
WORDS_TIME = ["23:00", "22:", "в 23", "ночь", "утра"]
WORDS_THREATS = ["испортим ки", "угроза", "коллекторы придут", "сосед", "подъезд"]

RULES_230FZ = {
    "law_title": "Федеральный закон № 230-ФЗ",
    "restrictions": [
        {"article": "ст. 7, ч. 5", "rule": "Запрет на взаимодействие в ночное время"},
        {"article": "ст. 6, ч. 1", "rule": "Запрет на угрозы и разглашение"},
        {"article": "ст. 7, ч. 4", "rule": "Ограничение частоты"}
    ]
}

# ==========================================
# 2. МОДЕЛИ ДАННЫХ (Pydantic)
# ==========================================
class Violation(BaseModel):
    article: str
    reason: str
    severity: str

class ComplianceResult(BaseModel):
    compliant: bool
    violations: List[Violation]
    recommendation: str

# ==========================================
# 3. БИЗНЕС-ЛОГИКА (Facade)
# ==========================================
def check_compliance(text: str) -> ComplianceResult:
    """
    Фасад: проверяет текст на соответствие 230-ФЗ.
    """
    if USE_MOCK:
        print("⚡ Режим MOCK (имитация)")
        time.sleep(0.5)
        text_lower = text.lower()
        violations = []

        # Проверка ночного времени
        if any(kw in text_lower for kw in WORDS_TIME):
            violations.append(Violation(article="ст. 7, ч. 5", reason="Взаимодействие в ночное время", severity="high"))

        # Проверка угроз и разглашения
        if any(kw in text_lower for kw in WORDS_THREATS):
            violations.append(Violation(article="ст. 6, ч. 1", reason="Угрозы или разглашение данных", severity="high"))

        return ComplianceResult(
            compliant=len(violations) == 0,
            violations=violations,
            recommendation="Текст содержит нарушения 230-ФЗ." if violations else "Текст корректен."
        )
    else:
        # Real API logic placeholder
        return ComplianceResult(compliant=True, violations=[], recommendation="API mode disabled.")

# ==========================================
# 4. ТЕСТЫ
# ==========================================
test_cases = [
    "Добрый день! Напоминаем об оплате до 15 числа.",
    "Оплати до вечера или приедем описывать имущество в 23:00!",
    "Ваш сосед Иван уже оплатил. Расскажите всем в подъезде."
]

print("\n🧪 ЗАПУСК ТЕСТОВ КОМПЛАЕНС-ЧЕКЕРА")
print("=" * 50)

for i, text in enumerate(test_cases, 1):
    print(f"\n📝 Тест {i}: \"{text}\"")
    result = check_compliance(text)

    status = "✅ PASS" if result.compliant else "❌ FAIL"
    print(f"Результат: {status}")

    for v in result.violations:
        print(f"   🚨 {v.severity.upper()} | {v.article}: {v.reason}")
    print(f"   💡 Рекомендация: {result.recommendation}")

print("\n" + "=" * 50)
print("🎉 Этап 1 завершен успешно!")


🧪 ЗАПУСК ТЕСТОВ КОМПЛАЕНС-ЧЕКЕРА

📝 Тест 1: "Добрый день! Напоминаем об оплате до 15 числа."
⚡ Режим MOCK (имитация)
Результат: ✅ PASS
   💡 Рекомендация: Текст корректен.

📝 Тест 2: "Оплати до вечера или приедем описывать имущество в 23:00!"
⚡ Режим MOCK (имитация)
Результат: ❌ FAIL
   🚨 HIGH | ст. 7, ч. 5: Взаимодействие в ночное время
   💡 Рекомендация: Текст содержит нарушения 230-ФЗ.

📝 Тест 3: "Ваш сосед Иван уже оплатил. Расскажите всем в подъезде."
⚡ Режим MOCK (имитация)
Результат: ❌ FAIL
   🚨 HIGH | ст. 6, ч. 1: Угрозы или разглашение данных
   💡 Рекомендация: Текст содержит нарушения 230-ФЗ.

🎉 Этап 1 завершен успешно!
